In [3]:
import time
import re
import pickle
import numpy as np
import torch
import faiss
from dotenv import load_dotenv
from groq import Groq

from scripts.utils import encode_text, MiniEmbeddingModel, load_all_knowledge
from scripts.rag_query import build_prompt

load_dotenv()
device = "cuda" if torch.cuda.is_available() else "cpu"


ModuleNotFoundError: No module named 'faiss'

In [ ]:
# Load knowledge
docs = load_all_knowledge("data/knowledge_base")

# Load BPE
with open("models/bpe_merges.pkl", "rb") as f:
    bpe_merges = pickle.load(f)
with open("models/bpe_vocab.pkl", "rb") as f:
    bpe_vocab = pickle.load(f)

# Load embedding model
model = MiniEmbeddingModel(len(bpe_vocab)).to(device)
model.load_state_dict(torch.load("models/sinhala_embedding_model.pt", map_location=device))
model.eval()

# Load FAISS
index = faiss.read_index("vectorstore/knowledge.index")

# Groq client
client = Groq(api_key=os.getenv("GROQ_API_KEY"))


In [ ]:
def get_frozen_context(query, top_k=3):
    q_ids = encode_text(query, bpe_merges, bpe_vocab).unsqueeze(0).to(device)
    with torch.no_grad():
        q_emb = model(q_ids).cpu().numpy()
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    _, indices = index.search(q_emb, top_k)
    return "\n\n".join([docs[i].page_content for i in indices[0]])


In [ ]:
def context_overlap(answer, context):
    a = set(answer.split())
    c = set(context.split())
    return len(a & c) / max(len(a), 1)

def sinhala_ratio(text):
    return len(re.findall(r"[\u0D80-\u0DFF]", text)) / max(len(text), 1)

def answer_length(answer):
    return len(answer.split())


In [ ]:
# Models to compare
MODELS = {
    "llama_70b": "llama-3.3-70b-versatile",
    "llama_8b": "llama-3.1-8b-instant"
}

# Test query
query = "ශ්‍රී ලංකාවේ බහුලව භාවිත වන ගව ආහාර කුමක්ද?"
context = get_frozen_context(query)
prompt = build_prompt(context, query)

for name, model_id in MODELS.items():
    start = time.time()
    response = client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=1024
    )
    latency = round(time.time() - start, 2)
    answer = response.choices[0].message.content.strip()
    
    print(f"\n🧠 Model: {name}")
    print("Latency (s):", latency)
    print("Context overlap:", round(context_overlap(answer, context), 3))
    print("Sinhala ratio:", round(sinhala_ratio(answer), 3))
    print("Answer length:", answer_length(answer))
    print("Answer:\n", answer)
